# Lab 3: Compaction â€” SH Reduction, Pruning, and Size Analysis

**Objectives:**
- Implement SH degree reduction and measure its impact on color accuracy
- Sweep opacity pruning thresholds and plot survival curves
- Remove floater Gaussians via scale thresholding
- Build a combined pipeline and quantify cumulative compression
- Analyze attribute entropy to estimate quantization headroom


In [ ]:
!pip install -q numpy matplotlib scipy
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy as scipy_entropy
%matplotlib inline
print('Ready.')

## Setup: Synthetic 3DGS Data (N=50,000)

In [ ]:
np.random.seed(42)
N = 50_000
positions      = np.random.randn(N, 3) * 2.0
scales         = np.exp(np.random.randn(N, 3) * 0.5 - 2.0)
quats          = np.random.randn(N, 4); quats /= np.linalg.norm(quats, axis=1, keepdims=True)
opacity_logits = np.random.randn(N) * 2.0 + 1.0
opacities      = 1.0 / (1.0 + np.exp(-opacity_logits))
sh_dc          = np.random.rand(N, 3)
sh_ac          = np.random.randn(N, 45) * 0.1

def size_mb(n, n_params):
    return n * n_params * 4 / 1e6

baseline_mb = size_mb(N, 59)
print(f'Baseline: {N:,} Gaussians x 59 params x 4 bytes = {baseline_mb:.1f} MB')

## Section 1: SH Degree Reduction

Reducing SH degree is the highest-leverage compaction strategy (SH = 76% of params).

In [ ]:
def sh_params(degree):
    """Total SH params per channel for given max degree."""
    return (degree + 1) ** 2

def params_for_degree(degree):
    """Total params per Gaussian for given SH degree."""
    return 3 + 3 + 4 + 1 + sh_params(degree) * 3  # pos+scale+rot+opacity + SH*3ch

# Evaluate SH degree reduction
def color_rmse_for_degree(sh_dc, sh_ac, degree, n_views=100):
    """
    Approximate RMSE from dropping SH beyond `degree`.
    We compare color evaluated with full SH vs. truncated SH at random view directions.
    This is a simplified model: we measure magnitude of dropped coefficients.
    """
    if degree == 3:
        return 0.0
    # Coefficients per channel in full SH: DC(1) + AC(15) = 16
    # DC is the first 1 coeff, orders 1-3 are the AC coeffs
    # sh_ac has shape (N, 45) = 15 coeffs/channel * 3 channels
    # Coefficients to drop: everything after degree
    keep_ac = (sh_params(degree) - 1)  # AC coefficients to keep per channel
    dropped_ac = sh_ac[:, keep_ac*3:]   # dropped coefficients
    return float(np.sqrt(np.mean(dropped_ac**2)))

print(f'{"Degree":>7} {"SH coeff/ch":>12} {"Total params":>13} {"Size (MB)":>11} {"Ratio":>7} {"Color RMSE":>12}')
print('-' * 65)
for deg in [0, 1, 2, 3]:
    total_p = params_for_degree(deg)
    mb = size_mb(N, total_p)
    ratio = baseline_mb / mb
    rmse = color_rmse_for_degree(sh_dc, sh_ac, deg)
    print(f'{deg:>7} {sh_params(deg):>12} {total_p:>13} {mb:>10.1f}  {ratio:>6.2f}x {rmse:>12.6f}')

## Section 2: Opacity-Based Pruning

In [ ]:
thresholds = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5]
keep_fracs, keep_counts, sizes_mb = [], [], []

for t in thresholds:
    mask = opacities >= t
    n_kept = mask.sum()
    keep_fracs.append(n_kept / N)
    keep_counts.append(n_kept)
    sizes_mb.append(size_mb(n_kept, 59))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogx(thresholds, [f*100 for f in keep_fracs], 'o-', color='steelblue', lw=2)
axes[0].set_xlabel('Opacity threshold'); axes[0].set_ylabel('Gaussians kept (%)')
axes[0].set_title('Survival Rate vs. Threshold'); axes[0].grid(True, alpha=0.3)
axes[0].axvline(0.05, color='red', ls='--', label='typical=0.05'); axes[0].legend()

axes[1].semilogx(thresholds, sizes_mb, 's-', color='coral', lw=2)
axes[1].set_xlabel('Opacity threshold'); axes[1].set_ylabel('Size (MB)')
axes[1].set_title('File Size vs. Threshold'); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f'{"Threshold":>10} {"Kept":>10} {"Fraction":>10} {"Size (MB)":>11} {"Ratio":>8}')
print('-' * 52)
for t, n, mb in zip(thresholds, keep_counts, sizes_mb):
    print(f'{t:>10.3f} {n:>10,} {n/N:>10.3f} {mb:>10.1f}  {baseline_mb/mb:>6.2f}x')

## Section 3: Scale-Based Floater Removal

In [ ]:
max_scales = scales.max(axis=1)  # largest axis of each Gaussian
p99 = np.percentile(max_scales, 99)
p95 = np.percentile(max_scales, 95)

mask_scale = max_scales < p99
print(f'99th percentile scale: {p99:.4f}')
print(f'Gaussians removed (>p99): {(~mask_scale).sum():,} ({(~mask_scale).mean()*100:.1f}%)')
print(f'Remaining: {mask_scale.sum():,}')

fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].hist(np.log10(max_scales), bins=60, color='slateblue', alpha=0.8)
ax[0].axvline(np.log10(p99), color='red', ls='--', label=f'p99={p99:.3f}')
ax[0].axvline(np.log10(p95), color='orange', ls='--', label=f'p95={p95:.3f}')
ax[0].set_xlabel('log10(max scale)'); ax[0].set_title('Max Scale Distribution'); ax[0].legend()

ax[1].hist(np.log10(max_scales[mask_scale]), bins=60, color='green', alpha=0.8)
ax[1].set_xlabel('log10(max scale)'); ax[1].set_title('After Floater Removal')
plt.tight_layout(); plt.show()

## Section 4: Combined Compaction Pipeline

In [ ]:
# Stage 1: SH reduction to degree 1
sh_degree = 1
params_after_sh = params_for_degree(sh_degree)
size_after_sh = size_mb(N, params_after_sh)

# Stage 2: Opacity pruning at 0.05
threshold = 0.05
mask_opacity = opacities >= threshold
N_after_opacity = mask_opacity.sum()
size_after_opacity = size_mb(N_after_opacity, params_after_sh)

# Stage 3: Scale floater removal at p99
mask_scale_p99 = max_scales < np.percentile(max_scales, 99)
mask_combined = mask_opacity & mask_scale_p99
N_final = mask_combined.sum()
size_final = size_mb(N_final, params_after_sh)

print('Compaction Pipeline Results:')
print(f'{"Stage":<35} {"N Gaussians":>12} {"Params/G":>10} {"Size (MB)":>11} {"Ratio":>8}')
print('-' * 80)
print(f'{"Baseline (degree 3)":<35} {N:>12,} {59:>10} {baseline_mb:>10.1f}  {1.0:>6.2f}x')
print(f'{"After SH deg-1 reduction":<35} {N:>12,} {params_after_sh:>10} {size_after_sh:>10.1f}  {baseline_mb/size_after_sh:>6.2f}x')
print(f'{"After opacity pruning (>0.05)":<35} {N_after_opacity:>12,} {params_after_sh:>10} {size_after_opacity:>10.1f}  {baseline_mb/size_after_opacity:>6.2f}x')
print(f'{"After scale pruning (p99)":<35} {N_final:>12,} {params_after_sh:>10} {size_final:>10.1f}  {baseline_mb/size_final:>6.2f}x')

## Section 5: Attribute Entropy Analysis

Shannon entropy tells us the minimum bits needed per value. Attributes with low entropy are more compressible.

In [ ]:
def empirical_entropy_bits(values, n_bins=256):
    """Estimate Shannon entropy in bits via histogram."""
    hist, _ = np.histogram(values.flatten(), bins=n_bins)
    hist = hist[hist > 0]
    probs = hist / hist.sum()
    return float(scipy_entropy(probs, base=2))

attrs = {
    'Position X':      positions[:, 0],
    'Scale X (log)':   np.log(scales[:, 0]),
    'Opacity (logit)': opacity_logits,
    'DC Color R':      sh_dc[:, 0],
    'SH AC coeff 0':   sh_ac[:, 0],
    'SH AC coeff 10':  sh_ac[:, 10],
}
print(f'{"Attribute":<22} {"Entropy (bits)":>15} {"float32 uses":>14} {"Savings":>10}')
print('-' * 64)
for name, vals in attrs.items():
    h = empirical_entropy_bits(vals)
    savings = 32 / h
    print(f'{name:<22} {h:>15.2f} {32:>14} {savings:>9.1f}x')

print('\nNote: Lower entropy = more compressible. SH AC coeffs have lowest entropy.')
print('Entropy < 8 bits means 8-bit quantization + entropy coding can store without loss.')

## Key Takeaways

- **SH degree reduction** gives 2â€“3Ã— parameter reduction with modest quality loss for diffuse scenes
- **Opacity pruning at 0.05** removes ~15â€“25% of Gaussians with negligible visual impact
- **Scale-based floater removal** at p99 removes <1% of Gaussians that would distort compression
- **Combined pipeline** achieves 3â€“5Ã— from compaction alone, before any encoding
- **Entropy analysis** shows SH AC coefficients have <6 bits of entropy â†’ 5-bit quantization loses almost nothing
- These compaction gains multiply with encoding gains (Module 5) â€” 3â€“5Ã— Ã— 6â€“7Ã— â‰ˆ 20â€“35Ã— total


---
## Part 2: Design Challenge

In Part 1 you *measured* an existing pipeline. Here you act as the engineer:
choose a scenario, write your strategy and predictions **before running any code**,
implement it using the Part 1 helpers, and then interpret the gap between what you
predicted and what actually happened.

### The Scenarios

**Scenario A — Diffuse Indoor Apartment** &nbsp;`N = 60,000 · target ≤ 5 MB`
An apartment walkthrough: matte walls, fabric sofas, wooden floors.
Low view-dependent effects; high floater count from training.

**Scenario B — Mixed Outdoor Plaza** &nbsp;`N = 50,000 · target ≤ 4 MB`
Concrete pavement (diffuse), metal benches (mild specular), glass storefront
(stronger specular). Outdoor scenes often contain large-scale background Gaussians
from sky or far geometry.

**Scenario C — Specular Marble Sculpture** &nbsp;`N = 30,000 · target ≤ 2 MB`
A polished marble figurine with strong specular highlights.
The target is intentionally tight. Your goal is to discover *empirically*
where the tools break down — not necessarily to meet the constraint.

---
All datasets are generated synthetically in the next cell.
The statistical properties of each (SH AC variance by order, opacity distribution,
scale distribution) are calibrated to match the described scene types.
All Part 1 helpers (`size_mb`, `params_for_degree`, `color_rmse_for_degree`, `empirical_entropy_bits`) are available.

In [ ]:
# ── Part 2: Scenario Datasets ────────────────────────────────────────────────
# SH AC variance is stratified by order (realistic: higher orders carry less energy
# in diffuse scenes, more in specular). Floaters use logit mean=-3.5 so they are
# solidly below the typical 0.05 opacity threshold.

# Scenario A: diffuse indoor apartment
np.random.seed(101); N_A = 60_000
sh_ac_A       = np.concatenate([np.random.randn(N_A,  9) * 0.040,   # order 1
                                  np.random.randn(N_A, 15) * 0.015,   # order 2
                                  np.random.randn(N_A, 21) * 0.005],  # order 3
                                 axis=1)
sh_dc_A       = np.random.rand(N_A, 3) * 0.4 + 0.3
_op_A         = np.concatenate([np.random.randn(30_000) * 1.2 + 3.0,   # solid
                                  np.random.randn(30_000) * 0.8 - 3.5]) # floaters (50%)
np.random.shuffle(_op_A)
opacities_A   = 1 / (1 + np.exp(-_op_A))
scales_A      = np.exp(np.random.randn(N_A, 3) * 0.5 - 2.5)

# Scenario B: mixed outdoor plaza
np.random.seed(202); N_B = 50_000
sh_ac_B       = np.concatenate([np.random.randn(N_B,  9) * 0.180,   # order 1
                                  np.random.randn(N_B, 15) * 0.080,   # order 2
                                  np.random.randn(N_B, 21) * 0.030],  # order 3
                                 axis=1)
sh_dc_B       = np.random.rand(N_B, 3) * 0.5 + 0.2
_op_B         = np.concatenate([np.random.randn(37_500) * 1.5 + 2.5,  # solid
                                  np.random.randn(12_500) * 0.8 - 3.5]) # floaters (25%)
np.random.shuffle(_op_B)
opacities_B   = 1 / (1 + np.exp(-_op_B))
scales_B      = np.exp(np.vstack([np.random.randn(47_500, 3) * 0.4 - 2.5,   # normal
                                    np.random.randn( 2_500, 3) * 0.3 + 1.0])) # sky/BG
np.random.shuffle(scales_B)

# Scenario C: specular marble sculpture
np.random.seed(303); N_C = 30_000
sh_ac_C       = np.concatenate([np.random.randn(N_C,  9) * 0.350,   # order 1 (strong)
                                  np.random.randn(N_C, 15) * 0.250,   # order 2
                                  np.random.randn(N_C, 21) * 0.150],  # order 3
                                 axis=1)
sh_dc_C       = np.random.rand(N_C, 3) * 0.3 + 0.6
_op_C         = np.concatenate([np.random.randn(27_000) * 1.0 + 3.0,  # solid
                                  np.random.randn( 3_000) * 0.7 - 3.5]) # floaters (10%)
np.random.shuffle(_op_C)
opacities_C   = 1 / (1 + np.exp(-_op_C))
scales_C      = np.exp(np.random.randn(N_C, 3) * 0.2 - 4.0)

# Summary table
print(f"{'Scenario':<32} {'N':>7} {'Baseline':>10} {'SH AC RMS':>11} {'Floaters<0.05':>14}")
print("-" * 78)
for label, N, sh_ac, ops in [
    ("A — Diffuse Indoor",    N_A, sh_ac_A, opacities_A),
    ("B — Mixed Outdoor",     N_B, sh_ac_B, opacities_B),
    ("C — Specular Marble",   N_C, sh_ac_C, opacities_C),
]:
    print(f"{label:<32} {N:>7,} {size_mb(N,59):>9.1f}MB "
          f"{np.sqrt(np.mean(sh_ac**2)):>11.5f} {(ops<0.05).mean()*100:>13.1f}%")

### Your Design Brief

**Write this before running any solution code.**
Add your answers as a new markdown cell (Insert → Text cell) or as comments below.

1. **Scenario chosen**: A / B / C
2. **SH AC RMS interpretation**: What does the value tell you about view dependence?
3. **Degree choice**: Which SH degree will you target and why?
4. **Pruning threshold**: What opacity threshold do you expect is safe? What retention rate do you predict?
5. **Scale pruning**: Will it help for your scenario? Why?
6. **Predicted final size**: Estimate MB after all stages.
7. **Will you meet the target?** If not, what would be needed?

In [ ]:
# YOUR DESIGN BRIEF
# (fill in your answers as comments before running the solution)

# Scenario chosen:
# SH AC RMS observation:
# Degree choice and rationale:
# Expected floater fraction / pruning threshold:
# Expected final size:
# Will the target be met?:

### Scenario A — Diffuse Indoor Apartment &nbsp;`target ≤ 5 MB`

Audit `sh_ac_A`, `opacities_A`, and `scales_A`. Write your design brief above, then implement below.

<details>
<summary>💡 Hint — Scenario A &nbsp;(read before the solution)</summary>
<br>
<p><strong>Start with the audit.</strong> Compute <code>np.sqrt(np.mean(sh_ac_A**2))</code>. How does this compare to Scenario B and C? A very low SH AC RMS means view-dependent color is minimal — you can safely reduce to a low SH degree with little visual cost.</p>
<p><strong>Opacity histogram.</strong> Re-use the histogram code from Section 2, but pass <code>_op_A</code> (the raw logits) or compute <code>opacities_A</code> directly. What fraction falls below 0.05? This fraction is your pruning headroom.</p>
<p><strong>Back-of-envelope.</strong> With 60K Gaussians, 23 params at degree 1, and the floater fraction you measured: expected post-pruning size = <code>60000 × (1 − floater_fraction) × 23 × 4 / 1e6</code> MB. Does that fall below 5 MB?</p>
<p><strong>Scale pruning.</strong> Plot <code>scales_A.max(axis=1)</code> on a log scale. Is there a long tail? If the distribution is roughly symmetric on a log scale, scale pruning will add little.</p>
</details>

<details>
<summary>✅ Example Solution — Scenario A &nbsp;(only open if stuck)</summary>
<br>
<p><em>Copy into a new code cell and run it.</em></p>
<pre style="background:#1e1e1e;color:#d4d4d4;padding:12px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.5"># SCENARIO A SOLUTION — Diffuse Indoor Apartment
# Strategy: SH degree 1 (low view dependence) + opacity pruning at 0.05
print(&quot;=&quot; * 60)
print(&quot;SCENARIO A: Diffuse Indoor Apartment&quot;)
print(&quot;Target: &lt;=5 MB&quot;)
print(&quot;=&quot; * 60)

# Step 1: Audit the dataset
baseline_A  = size_mb(N_A, 59)
floater_A   = (opacities_A &lt; 0.05).mean()
sh_rms_A    = np.sqrt(np.mean(sh_ac_A ** 2))
print(f&quot;\nBaseline : {N_A:,} Gaussians = {baseline_A:.1f} MB&quot;)
print(f&quot;SH AC RMS: {sh_rms_A:.5f}  &lt;- very low =&gt; diffuse =&gt; degree 1 is safe&quot;)
print(f&quot;Floaters (&lt;0.05 opacity): {floater_A*100:.1f}%  &lt;- pruning headroom&quot;)

# Step 2: SH degree reduction
deg_A = 1
p_A   = params_for_degree(deg_A)
sz_sh = size_mb(N_A, p_A)
rmse_A = color_rmse_for_degree(sh_dc_A, sh_ac_A, deg_A)
print(f&quot;\nAfter SH degree {deg_A}: {p_A} params/G =&gt; {sz_sh:.1f} MB  (RMSE {rmse_A:.5f})&quot;)

# Step 3: Opacity pruning
t_A        = 0.05
mask_op_A  = opacities_A &gt;= t_A
sz_op_A    = size_mb(mask_op_A.sum(), p_A)
print(f&quot;After opacity pruning (&gt;={t_A}): {mask_op_A.sum():,} kept =&gt; {sz_op_A:.2f} MB&quot;)

# Step 4: Scale pruning (check the distribution first)
max_sc_A   = scales_A.max(axis=1)
p99_A      = np.percentile(max_sc_A, 99)
mask_fin_A = mask_op_A &amp; (max_sc_A &lt; p99_A)
sz_fin_A   = size_mb(mask_fin_A.sum(), p_A)
print(f&quot;After scale pruning (p99={p99_A:.4f}): {mask_fin_A.sum():,} kept =&gt; {sz_fin_A:.2f} MB&quot;)

# Result
print(f&quot;\n{&#x27;RESULT&#x27;:=&lt;60}&quot;)
print(f&quot;  Baseline  : {baseline_A:.1f} MB&quot;)
print(f&quot;  Final     : {sz_fin_A:.2f} MB  (target &lt;=5 MB) {&#x27;ACHIEVED&#x27; if sz_fin_A&lt;=5 else &#x27;NOT ACHIEVED&#x27;}&quot;)
print(f&quot;  Color RMSE: {rmse_A:.5f}  (low -- diffuse scene tolerates degree-1 well)&quot;)
print(f&quot;  Compression: {baseline_A/sz_fin_A:.1f}x&quot;)
print(&quot;\nKey insight: large floater population (37-38%) drove most of the gain.&quot;)
print(&quot;SH reduction alone gave 2.6x; pruning added another ~1.9x on top.&quot;)</pre>
</details>

In [ ]:
# Your Scenario A implementation


### Scenario B — Mixed Outdoor Plaza &nbsp;`target ≤ 4 MB`

Audit `sh_ac_B`, `opacities_B`, and `scales_B`. Decide on your strategy before opening the solution.

<details>
<summary>💡 Hint — Scenario B &nbsp;(read before the solution)</summary>
<br>
<p><strong>The SH AC RMS is moderate.</strong> Compare the RMSE from <code>color_rmse_for_degree</code> for degree 1 vs degree 2. The gap tells you how much quality you lose by choosing degree 1. Given the target, is the extra 15 params/Gaussian for degree 2 worth it?</p>
<p><strong>The scale distribution matters here.</strong> Run the scale-distribution plot from Section 3 on <code>scales_B</code>. Are there Gaussians with max scale 10–100× larger than the median? If so, scale pruning at p99 will remove them at essentially zero quality cost.</p>
<p><strong>Sequence matters.</strong> Apply SH reduction → opacity pruning → scale pruning in that order. Audit the size after each stage. If you are still over target after opacity pruning, check whether the scale outliers are large enough to close the gap.</p>
</details>

<details>
<summary>✅ Example Solution — Scenario B &nbsp;(only open if stuck)</summary>
<br>
<p><em>Copy into a new code cell and run it.</em></p>
<pre style="background:#1e1e1e;color:#d4d4d4;padding:12px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.5"># SCENARIO B SOLUTION — Mixed Outdoor Plaza
# Strategy: SH degree 1 + opacity pruning 0.05 + scale pruning at p99
print(&quot;=&quot; * 60)
print(&quot;SCENARIO B: Mixed Outdoor Plaza&quot;)
print(&quot;Target: &lt;=4 MB&quot;)
print(&quot;=&quot; * 60)

# Step 1: Audit
baseline_B   = size_mb(N_B, 59)
floater_B    = (opacities_B &lt; 0.05).mean()
sh_rms_B     = np.sqrt(np.mean(sh_ac_B ** 2))
max_sc_B     = scales_B.max(axis=1)
p99_B        = np.percentile(max_sc_B, 99)
p999_B       = np.percentile(max_sc_B, 99.9)
print(f&quot;\nBaseline : {N_B:,} Gaussians = {baseline_B:.1f} MB&quot;)
print(f&quot;SH AC RMS: {sh_rms_B:.5f}  &lt;- moderate =&gt; degree 1 is reasonable&quot;)
print(f&quot;Floaters (&lt;0.05): {floater_B*100:.1f}%&quot;)
print(f&quot;Scale p99={p99_B:.3f}  p99.9={p999_B:.3f}  &lt;- wide tail =&gt; scale pruning helps&quot;)

# Step 2: Compare degree 1 vs degree 2 quality
print(&quot;\nSH degree quality tradeoff:&quot;)
for deg in [1, 2]:
    rmse = color_rmse_for_degree(sh_dc_B, sh_ac_B, deg)
    print(f&quot;  degree {deg}: RMSE={rmse:.5f}  params={params_for_degree(deg)}/G&quot;)

# Use degree 1 (acceptable RMSE, better compression)
deg_B = 1
p_B   = params_for_degree(deg_B)
rmse_B = color_rmse_for_degree(sh_dc_B, sh_ac_B, deg_B)
sz_sh_B = size_mb(N_B, p_B)
print(f&quot;\nAfter SH degree {deg_B}: {sz_sh_B:.1f} MB  (RMSE {rmse_B:.5f})&quot;)

# Step 3: Opacity pruning
mask_op_B  = opacities_B &gt;= 0.05
sz_op_B    = size_mb(mask_op_B.sum(), p_B)
print(f&quot;After opacity pruning (&gt;=0.05): {mask_op_B.sum():,} kept =&gt; {sz_op_B:.2f} MB&quot;)

# Step 4: Scale pruning — especially valuable here (5% large outliers)
mask_fin_B = mask_op_B &amp; (max_sc_B &lt; p99_B)
sz_fin_B   = size_mb(mask_fin_B.sum(), p_B)
print(f&quot;After scale pruning (p99): {mask_fin_B.sum():,} kept =&gt; {sz_fin_B:.2f} MB&quot;)

# Result
print(f&quot;\n{&#x27;RESULT&#x27;:=&lt;60}&quot;)
print(f&quot;  Baseline  : {baseline_B:.1f} MB&quot;)
print(f&quot;  Final     : {sz_fin_B:.2f} MB  (target &lt;=4 MB) {&#x27;ACHIEVED&#x27; if sz_fin_B&lt;=4 else &#x27;NOT ACHIEVED&#x27;}&quot;)
print(f&quot;  Color RMSE: {rmse_B:.5f}&quot;)
print(f&quot;  Compression: {baseline_B/sz_fin_B:.1f}x&quot;)
print(&quot;\nNote: scale pruning contributed meaningfully here (large-scale outliers&quot;)
print(&quot;from sky/background geometry). On Scenario A they added almost nothing.&quot;)</pre>
</details>

In [ ]:
# Your Scenario B implementation


### Scenario C — Specular Marble Sculpture &nbsp;`target ≤ 2 MB`

> ⚠️ This scenario is **intentionally constrained**. Your goal is not necessarily to meet the target — it is to discover empirically why the constraint cannot be satisfied with these tools alone, and to articulate what would be needed.

Audit `sh_ac_C`, `opacities_C`, and `scales_C`. Write your design brief, then test your best available strategy.

<details>
<summary>💡 Hint — Scenario C &nbsp;(read before the solution)</summary>
<br>
<p><strong>Read the SH AC RMS carefully.</strong> It is substantially higher than both A and B. What does this mean for degree choice? Use <code>color_rmse_for_degree</code> at degree 0, 1, and 2 to quantify the tradeoff.</p>
<p><strong>Think about the constraint geometry.</strong> The minimum possible size (degree 0, no pruning at all) is <code>size_mb(N_C, params_for_degree(0))</code>. Is it above or below the 2 MB target? What does this tell you about what is achievable?</p>
<p><strong>The floater fraction is small (~8%).</strong> How much pruning headroom does that give you? For each degree, compute the post-pruning size and RMSE. Is there any combination that satisfies both constraints?</p>
<p><strong>The post-mortem is the point.</strong> Identify the specific combination that gets closest to meeting both constraints, explain the gap, and state what structural change would close it.</p>
</details>

<details>
<summary>✅ Example Solution — Scenario C &nbsp;(only open if stuck)</summary>
<br>
<p><em>Copy into a new code cell and run it.</em></p>
<pre style="background:#1e1e1e;color:#d4d4d4;padding:12px;border-radius:6px;overflow-x:auto;font-size:12px;line-height:1.5"># SCENARIO C SOLUTION — Specular Marble Sculpture
# Intentional lesson: attribute reduction + pruning cannot meet BOTH constraints
print(&quot;=&quot; * 60)
print(&quot;SCENARIO C: Specular Marble Sculpture&quot;)
print(&quot;Target: &lt;=2 MB  (intentionally tight)&quot;)
print(&quot;=&quot; * 60)

# Step 1: Audit — read the signals before committing to a strategy
baseline_C   = size_mb(N_C, 59)
floater_C    = (opacities_C &lt; 0.05).mean()
sh_rms_C     = np.sqrt(np.mean(sh_ac_C ** 2))
min_possible = size_mb(N_C, params_for_degree(0))  # degree 0, no pruning
print(f&quot;\nBaseline : {N_C:,} Gaussians = {baseline_C:.1f} MB&quot;)
print(f&quot;SH AC RMS: {sh_rms_C:.4f}  &lt;- HIGH =&gt; strong specular =&gt; reducing degree is costly&quot;)
print(f&quot;Floaters (&lt;0.05): {floater_C*100:.1f}%  &lt;- very few =&gt; pruning headroom is small&quot;)
print(f&quot;Min possible (degree 0, no pruning): {min_possible:.2f} MB&quot;)

# Step 2: Evaluate all available degrees after pruning
mask_op_C  = opacities_C &gt;= 0.05
max_sc_C   = scales_C.max(axis=1)
mask_fin_C = mask_op_C &amp; (max_sc_C &lt; np.percentile(max_sc_C, 99))
N_fin_C    = mask_fin_C.sum()
print(f&quot;\nGaussians after max pruning: {N_fin_C:,} ({N_fin_C/N_C*100:.1f}% of original)&quot;)
print(f&quot;\n{&#x27;Degree&#x27;:&gt;7} {&#x27;Params/G&#x27;:&gt;9} {&#x27;Size (MB)&#x27;:&gt;11} {&#x27;RMSE&#x27;:&gt;10} {&#x27;Size OK&#x27;:&gt;9} {&#x27;RMSE OK&#x27;:&gt;9}&quot;)
print(&quot;-&quot; * 60)
for deg in [0, 1, 2, 3]:
    p    = params_for_degree(deg)
    sz   = size_mb(N_fin_C, p)
    rmse = color_rmse_for_degree(sh_dc_C, sh_ac_C, deg)
    print(f&quot;{deg:&gt;7}  {p:&gt;9}  {sz:&gt;10.2f}  {rmse:&gt;10.4f}  &quot;
          f&quot;{&#x27;Yes&#x27; if sz&lt;=2.0 else &#x27;No&#x27;:&gt;9}  {&#x27;Yes&#x27; if rmse&lt;0.05 else &#x27;No&#x27;:&gt;9}&quot;)

print()
print(&quot;OBSERVATION: No available degree satisfies BOTH constraints.&quot;)
print(&quot;  - degree 0: meets size target but RMSE is unacceptably high (specular)&quot;)
print(&quot;  - degree 1+: better quality but misses the 2 MB size target&quot;)
print()
print(&quot;CONCLUSION: The &lt;=2 MB target with acceptable quality requires either:&quot;)
print(&quot;  (a) Structural compression — ScaffoldGS + HAC (Module 5 / R3)&quot;)
print(&quot;  (b) Post-hoc quantization + entropy coding (Module 5)&quot;)
print()
print(&quot;This is not a failure — it is the correct engineering diagnosis.&quot;)
print(&quot;Recognizing when a constraint requires architectural changes, not just&quot;)
print(&quot;parameter tweaks, is the core skill this scenario is designed to build.&quot;)</pre>
</details>

In [ ]:
# Your Scenario C implementation


### Post-Mortem Reflection

After running your chosen scenario, answer these in a text cell:

1. **Largest prediction miss** — Which stage deviated most from your pre-run estimate? Why?
2. **Scene type signal** — How did the SH AC RMS guide (or mislead) your degree choice?
3. **Scenario C lesson** — Why can attribute reduction + pruning not meet ≤ 2 MB *and* low RMSE on a specular scene? What specifically does ScaffoldGS (R3) do that these tools cannot?
4. **Rule of thumb you'd carry forward** — Complete: *"For a scene with SH AC RMS above ___, I would not reduce below SH degree ___ because ..."*